# Chapter 12: Demystifying Machine Learning

## 12.3 Important Python Packages

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms
from datasets import load_dataset
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [ ]:
notebook_dir = Path.cwd()
out_dir = notebook_dir / "models"
out_dir.mkdir(parents=True, exist_ok=True)

## 12.4 Data Acquisition and Preprocessing

In [ ]:
dataset = load_dataset("blanchon/EuroSAT_RGB")
print(dataset)

In [ ]:
# Class names for EuroSAT
CLASS_NAMES = ["Annual Crop", "Forest", "Herbaceous Vegetation", "Highway",
               "Industrial", "Pasture", "Permanent Crop", "Residential",
               "River", "Sea Lake"]

In [ ]:
dataset["validation"][0]["image"]

In [ ]:
dataset["validation"][0]["label"]

In [ ]:
sample_labels = np.array(dataset["validation"]["label"])

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
axes = axes.ravel()

for ax, class_idx in zip(axes, range(len(CLASS_NAMES))):
    idxs = np.where(sample_labels == class_idx)[0]
    class_sample = dataset["validation"][int(idxs[0])]

    ax.imshow(class_sample["image"])
    ax.axis("off")
    ax.set_title(CLASS_NAMES[class_idx], fontsize=13)

plt.tight_layout()
plt.show()


### 12.4.1 Class Balancing

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, split in enumerate(["train", "validation", "test"]):
    labels = [item["label"] for item in dataset[split]]
    counts = np.bincount(labels)

    axes[idx].bar(range(10), counts)
    axes[idx].set_xticks(range(10))
    axes[idx].set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    axes[idx].set_ylabel("Count")
    axes[idx].set_title(f"{split.capitalize()} Set (n={len(labels)})")

plt.tight_layout()
plt.show()

### 12.4.1 Loading, Transforming, and Batch Size

In [ ]:
class EuroSATDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"].convert("RGB")
        label = item["label"]
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], \
        [0.229, 0.224, 0.225]),
    ])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], \
        [0.229, 0.224, 0.225]),
    ])

In [ ]:
train_dataset = EuroSATDataset(dataset["train"], train_transform)
val_dataset = EuroSATDataset(dataset["validation"], val_transform)
test_dataset = EuroSATDataset(dataset["test"], val_transform)

In [ ]:
BATCH_SIZE = 64

In [ ]:
# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, \
    shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, \
    shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, \
    shuffle=False, num_workers=0)

## 12.5 Model Initialization

In [ ]:
# Configuration
EPOCHS = 10
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def build_model(num_classes=10, freeze_backbone=True, model_name="resnet50"):
    # Select model architecture
    if model_name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    elif model_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    elif model_name == "resnet101":
        model = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    return model.to(DEVICE)

## 12.6 Model Training

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return running_loss / len(loader), 100. * correct / total, \
        all_preds, all_labels

In [ ]:
model_name = "resnet50"
filename_hist_all = f"all_model_{model_name}_history.csv"

model = build_model(num_classes=10, \
    freeze_backbone=True, model_name=model_name)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

best_val_acc = 0.0
history_resnet = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, _, _ = validate(model, val_loader, criterion)
    print(f" - Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f" - Valid Loss: {val_loss:.4f} | Valid Acc: {val_acc:.2f}%")

    history_resnet["train_loss"].append(train_loss)
    history_resnet["train_acc"].append(train_acc)
    history_resnet["val_loss"].append(val_loss)
    history_resnet["val_acc"].append(val_acc)
    df = pd.DataFrame(history_resnet)

    # Save all training history for learning curves
    df.to_csv(out_dir / filename_hist_all, index=False)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Save best model checkpoint
        filename_model = f"best_model_{model_name}.pth"
        torch.save(model.state_dict(), out_dir / filename_model)

## 12.7 Model Evaluation and Interpretation

In [ ]:
def plot_training_history(history):

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
    
    epochs = range(1, len(history["train_loss"]) + 1)
    
    # Loss plot
    ax1.plot(epochs, history["train_loss"], "b-", label="Train Loss", linewidth=2)
    ax1.plot(epochs, history["val_loss"], "r-", label="Val Loss", linewidth=2)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training and Validation Loss")
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # Accuracy plot
    ax2.plot(epochs, history["train_acc"], "b-", label="Train Acc", linewidth=2)
    ax2.plot(epochs, history["val_acc"], "r-", label="Val Acc", linewidth=2)
    ax2.set_xlabel("Epoch")
    
    ax2.set_title("Training and Validation Accuracy")
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
history_resnet = pd.read_csv(out_dir / f"all_model_{model_name}_history.csv")
plot_training_history(history_resnet)

In [ ]:
model.load_state_dict(torch.load(out_dir /f"best_model_{model_name}.pth", weights_only=True, map_location=DEVICE))
test_loss, test_acc, test_preds, test_labels = validate(model, test_loader, criterion)
print(f"Best validation accuracy: {best_val_acc:.2f}%")
print(f"Final test accuracy: {test_acc:.2f}%")

In [ ]:
def plot_per_class_accuracy(test_labels, test_preds, CLASS_NAMES):
    class_name_labels = [name.replace(" ", "\n") for name in CLASS_NAMES]

    # Calculate accuracies for each label
    accuracies = []
    for class_idx in range(10):
        mask = np.array(test_labels) == class_idx
        class_acc = (np.array(test_preds)[mask] == class_idx).mean() * 100
        accuracies.append(class_acc)
    
    # Make bar plot
    plt.figure(figsize=(10, 5))
    bars = plt.bar(range(10), accuracies, color="steelblue")
    plt.xticks(range(10), class_name_labels, rotation=45, ha="right")
    plt.ylabel("Accuracy (%)")
    plt.title("Per-Class Test Accuracy")
    plt.ylim([0, 100])
    plt.tight_layout()
    plt.show()

In [ ]:
plot_per_class_accuracy(test_labels, test_preds, CLASS_NAMES)

In [ ]:
print(classification_report(test_labels, test_preds,\
    target_names=CLASS_NAMES))

In [ ]:
class_name_labels = [name.replace(" ", "\n") for name in CLASS_NAMES]
ConfusionMatrixDisplay.from_predictions(test_labels, test_preds, \
    display_labels=class_name_labels, xticks_rotation="vertical", cmap="Blues")

In [ ]:
def plot_predictions(model, test_loader, DEVICE, class_names, \
        target_class=None, seed=38):
    # 1. Set model to evaluation mode
    model.eval()
    all_images = []
    all_preds = []
    all_labels = []
    
    # 2. Collect predictions from test dataset
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = outputs.max(1)
            
            all_images.extend(images.cpu())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    all_images = torch.stack(all_images)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
        
    # 3. Define the four prediction types (order: TP, FN, FP, TN)
    target_idx = class_names.index(target_class)

    prediction_types = {
        "TP": {
            "mask": (all_preds == target_idx) & (all_labels == target_idx), \
            "color": "blue"
        },
        "FN": {
            "mask": (all_preds != target_idx) & (all_labels == target_idx), \
            "color": "darkorange"
        },
        "FP": {
            "mask": (all_preds == target_idx) & (all_labels != target_idx), \
            "title": f"False Positive",
            "color": "darkorange"
        },
        "TN": {
            "mask": (all_preds != target_idx) & (all_labels != target_idx) \
            & (all_preds == all_labels), "color": "blue"
        }
    }
    
    # 4. Create Plot
    fig, axes = plt.subplots(2, 2, figsize=(6, 6))
    axes = axes.flatten()
    
    plot_idx = 0
    
    for pred_type, info in prediction_types.items():
        indices = np.where(info["mask"])[0]
        
        # Select one random image
        np.random.seed(seed)
        selected_idx = np.random.choice(indices)
        
        # Denormalize image
        img = all_images[selected_idx].numpy()
        img = np.transpose(img, (1, 2, 0))
        img = img * np.array([0.229, 0.224, 0.225]) + \
            np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)
        
        axes[plot_idx].imshow(img)
        
        pred_label = class_names[all_preds[selected_idx]]
        true_label = class_names[all_labels[selected_idx]]
        
        axes[plot_idx].set_title(\
            f"Predicted: {pred_label}\nActual: {true_label}", 
            fontsize=9, color=info["color"])
        axes[plot_idx].axis("off")

        plot_idx += 1
    
    plt.tight_layout()
    plt.show()

plot_predictions(model, test_loader, DEVICE, CLASS_NAMES, \
    target_class="Annual Crop", seed=37)

In [ ]:
plot_predictions(model, test_loader, DEVICE, CLASS_NAMES, \
    target_class="Highway", seed=41)

In [ ]:
# def plot_predictions(model, test_loader, criterion, DEVICE, class_names, target_class=None, seed=38):
#     # 1. Collect predictions using validate function
#     criterion = torch.nn.CrossEntropyLoss()
    
#     _, _, all_preds, all_labels = validate(model, test_loader, criterion)
#     all_preds = np.array(all_preds)
#     all_labels = np.array(all_labels)
    
#     # 2. Collect images
#     all_images = []
#     with torch.no_grad():
#         for images, labels in test_loader:
#             all_images.extend(images.cpu())
#     all_images = torch.stack(all_images)
    
#     # 3. Define the four prediction types
#     target_idx = class_names.index(target_class)
    
#     prediction_types = {
#         "True Positive": {
#             "mask": (all_preds == target_idx) & (all_labels == target_idx),
#             "color": "blue"
#         },
#         "False Negative": {
#             "mask": (all_preds != target_idx) & (all_labels == target_idx),
#             "color": "darkorange"
#         },
#         "False Positive": {
#             "mask": (all_preds == target_idx) & (all_labels != target_idx),
#             "color": "darkorange"
#         },
#         "True Negative": {
#             "mask": (all_preds != target_idx) & (all_labels != target_idx) & (all_preds == all_labels),
#             "color": "blue"
#         }
#     }
    
#     # 4. Create Plot
#     fig, axes = plt.subplots(2, 2, figsize=(6, 6))
#     axes = axes.flatten()
    
#     for plot_idx, (pred_type, info) in enumerate(prediction_types.items()):
#         indices = np.where(info["mask"])[0]
        
#         # Select one random image
#         np.random.seed(seed + plot_idx)  # Vary seed per subplot
#         selected_idx = np.random.choice(indices)
        
#         # Denormalize image
#         img = all_images[selected_idx].numpy()
#         img = np.transpose(img, (1, 2, 0))
#         img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
#         img = np.clip(img, 0, 1)
        
#         axes[plot_idx].imshow(img)
        
#         pred_label = class_names[all_preds[selected_idx]]
#         true_label = class_names[all_labels[selected_idx]]
        
#         axes[plot_idx].set_title(f"{pred_type}\nPredicted: {pred_label}\nActual: {true_label}", 
#                     fontsize=9)
#         axes[plot_idx].axis("off")
    
#     plt.tight_layout()
#     plt.show()

# # Usage
# plot_predictions(model, test_loader, criterion, DEVICE, CLASS_NAMES, target_class="Annual Crop", seed=41)

## 12.8 Model Refinement

Note: Code below is used to generate the various models, but not used in the book chapter.

In [ ]:
# MODEL_NAMES = ["resnet50"]
# LEARNING_RATES = [0.001, 0.0001]
# EPOCHS = [5, 10, 30, 50]

# for model_name in MODEL_NAMES:
#     for learning_rate in LEARNING_RATES:
#         for tot_epoch in EPOCHS:
#             print(f"\nTraining {model_name} with learning rate {learning_rate}")
#             model = build_model(num_classes=10, freeze_backbone=True, model_name=model_name)
#             criterion = nn.CrossEntropyLoss()
#             optimizer = optim.Adam(model.fc.parameters(), lr=learning_rate)

#             best_val_acc = 0.0
#             history_resnet = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

#             for epoch in range(tot_epoch):
#                 print(f"Epoch {epoch+1}/{tot_epoch}")
#                 train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
#                 val_loss, val_acc, _, _ = validate(model, val_loader, criterion)
#                 print(f" - Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
#                 print(f" - Valid Loss: {val_loss:.4f} | Valid Acc: {val_acc:.2f}%")

#                 history_resnet["train_loss"].append(train_loss)
#                 history_resnet["train_acc"].append(train_acc)
#                 history_resnet["val_loss"].append(val_loss)
#                 history_resnet["val_acc"].append(val_acc)

#                 # Save model weights - only saving final epoch
#                 # if val_acc > best_val_acc:
#                 # best_val_acc = val_acc
#                 filename_model = f"final_model_{model_name}_{str(learning_rate).replace(".", "_")}_EPOCH_{str(tot_epoch)}.pth"
#                 torch.save(model.state_dict(), out_dir / filename_model)

#             # Save data for learning curves
#             df = pd.DataFrame(history_resnet)
#             filename_hist = f"final_model_{model_name}_{str(learning_rate).replace(".", "_")}_EPOCH_{str(tot_epoch)}_history.csv"
#             df.to_csv(out_dir / filename_hist, index=False)

In [ ]:
model_configs_arch = [
    {"arch": "resnet18", "lr": 0.001, "epochs": 10, \
    "path": out_dir/"best_model_resnet18_0_001.pth", \
    "hist": out_dir/"best_model_resnet18_0_001_history.csv"},

    {"arch": "resnet18", "lr": 0.0001, "epochs": 10, \
    "path": out_dir/"best_model_resnet18_0_0001.pth", \
    "hist": out_dir/"best_model_resnet18_0_0001_history.csv"},

    {"arch": "resnet50", "lr": 0.001, "epochs": 10, \
    "path": out_dir/"best_model_resnet50_0_001.pth", \
    "hist": out_dir/"best_model_resnet50_0_001_history.csv"},

    {"arch": "resnet50", "lr": 0.0001, "epochs": 10, \
    "path": out_dir/"best_model_resnet50_0_0001.pth", \
    "hist": out_dir/"best_model_resnet50_0_0001_history.csv"},
    {"arch": "resnet101", "lr": 0.001, "epochs": 10, \
    "path": out_dir/"best_model_resnet101_0_001.pth", \
    "hist": out_dir/"best_model_resnet101_0_001_history.csv"},

    {"arch": "resnet101", "lr": 0.0001, "epochs": 10, \
    "path": out_dir/"best_model_resnet101_0_0001.pth", \
    "hist": out_dir/"best_model_resnet101_0_0001_history.csv"}
]

In [ ]:
# Usage in your comparison loop
def evaluate_models(model_configs):
    results = []

    for info in model_configs:
        # Rebuild model
        model = build_model(num_classes=10, freeze_backbone=False, model_name=info["arch"])
        
        # Load your trained weights
        model.load_state_dict(torch.load(info["path"], weights_only=True, map_location=DEVICE))
        
        # Reuse your validate function
        __, __, test_preds, test_labels = validate(model, test_loader, criterion)

        accuracy = accuracy_score(test_labels, test_preds)
        precision = precision_score(test_labels, test_preds, average="weighted")
        recall = recall_score(test_labels, test_preds, average="weighted")
        f1 = f1_score(test_labels, test_preds, average="weighted")
        
        results.append({
            "model": info["arch"],
            "learning rate": info["lr"],
            "training epochs": info["epochs"],
            "accuracy": round(accuracy,2),
            "precision": round(precision,2),
            "recall": round(recall,2),
            "f1_score": round(f1,2)
        })

    # Convert to DataFrame
    df_results = pd.DataFrame(results)

    return df_results

In [ ]:
df_results_arch = evaluate_models(model_configs_arch)
df_results_arch

In [ ]:
model_configs_epoch = [
    {"arch": "resnet50", "lr": 0.001, "epochs": 5,
    "path": out_dir/"final_model_resnet50_0_001_EPOCH_5.pth",
    "hist": out_dir/"final_model_resnet50_0_001_EPOCH_5_history.csv"},

    {"arch": "resnet50", "lr": 0.001, "epochs": 10,
    "path": out_dir/"final_model_resnet50_0_001_EPOCH_10.pth",
    "hist": out_dir/"final_model_resnet50_0_001_EPOCH_10_history.csv"},

    {"arch": "resnet50", "lr": 0.001, "epochs": 30, 
    "path": out_dir/"final_model_resnet50_0_001_EPOCH_30.pth", 
    "hist": out_dir/"final_model_resnet50_0_001_EPOCH_30_history.csv"},

    {"arch": "resnet50", "lr": 0.001, 
    "epochs": 50, "path": out_dir/"final_model_resnet50_0_001_EPOCH_50.pth", 
    "hist": out_dir/"final_model_resnet50_0_001_EPOCH_50_history.csv"}
    ]

In [ ]:
df_results_epoch = evaluate_models(model_configs_epoch)
df_results_epoch

In [ ]:
for info in model_configs_epoch:
    filename_hist = info["hist"]
    history = pd.read_csv(filename_hist)
    
    print(filename_hist)
    plot_training_history(history)

In [ ]:
for info in model_configs_epoch:

    model.load_state_dict(torch.load(info["path"], weights_only=True, map_location=DEVICE))
    test_loss, test_acc, test_preds, test_labels = validate(model, test_loader, criterion)
    print(info["path"])
    print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES))

Code below not used in book for brevity, but is useful for looking at the confusion matrices of each model

In [ ]:
# for info in model_configs_epoch:
#     model.load_state_dict(torch.load(info["path"], weights_only=True))
#     test_loss, test_acc, test_preds, test_labels = validate(model, test_loader, criterion)
#     print(info["path"])

#     class_name_labels = [name.replace(" ", "\n") for name in CLASS_NAMES]
#     ConfusionMatrixDisplay.from_predictions(test_labels, test_preds, display_labels=class_name_labels, xticks_rotation="vertical", cmap="Blues")